<a href="https://colab.research.google.com/github/dannyngweekiat/Introduction-to-Machine-Vision/blob/main/notebooks/08_mini_ai_vision_project.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a> <a href="https://github.com/dannyngweekiat/Introduction-to-Machine-Vision/blob/main/notebooks/08_mini_ai_vision_project.ipynb" target="_blank"><img src="https://img.shields.io/badge/View%20on-GitHub-181717?style=flat-square&amp;logo=github&amp;logoColor=white" alt="View on GitHub"></a>

# Lesson 8: Mini AI Vision Project

## Your mission
Choose one simple vision project, customise it, test it fairly, and explain a limitation.

**Key words:** test · customise · limitation

Each project below is independent. Pick one section, run its code cell, then make one change at a time.


## Choose a project section

- **Project A — Red motorbike highlighter:** keep one colour from a photograph.
- **Project B — Tent-or-boat recogniser:** train KNN to label tiny object icons.
- **Project C — Four-picture image classifier:** compare a model's top three labels for a busy collage.
- **Project D — Street-scene detector:** find a bus, people, and traffic lights.

You do not need to change a project switch. Run only the section you want to explore.


## Project A: Red motorbike highlighter

Use an HSV colour mask to keep the bright red paint on a motorbike.


In [ ]:
# Import OpenCV for colour conversion and masking.
import cv2
# Import NumPy for the colour-range arrays.
import numpy as np
# Import Matplotlib to display the results.
import matplotlib.pyplot as plt
# Load a ready-made stereo photograph of a motorbike.
from skimage import data

# Choose the left photograph from the motorbike image pair.
image = data.stereo_motorcycle()[0]
# Convert RGB colours into HSV, which is easier to select by colour.
hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
# Set the lowest red HSV values to keep.
lower = np.array([0, 180, 100])
# Set the highest red HSV values to keep.
upper = np.array([7, 255, 255])
# Make a black-and-white mask for strongly red motorbike paint.
mask = cv2.inRange(hsv, lower, upper)
# Keep only the pixels selected by the red-paint mask.
result = cv2.bitwise_and(image, image, mask=mask)

# Create three side-by-side spaces for a clear comparison.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Show the original motorbike photograph.
axes[0].imshow(image)
# Label the original photograph.
axes[0].set_title('Original motorbike')
# Show the black-and-white red-paint mask.
axes[1].imshow(mask, cmap='gray')
# Label the colour selection mask.
axes[1].set_title('Red paint mask')
# Show the red motorbike pixels on their own.
axes[2].imshow(result)
# Label the final colour result.
axes[2].set_title('Red pixels kept')
# Repeat the same tidy-up step for every plot.
for ax in axes:
    # Hide axes because they do not help us read this image.
    ax.axis('off')
# Space the plots so titles stay readable.
fig.tight_layout()
# Show how many pixels matched the red-colour rule.
print('Red pixels:', np.count_nonzero(mask))


## Project B: Tent-or-boat recogniser

Train a small KNN model to tell tiny tent icons from sailboat icons.


In [ ]:
# Import Matplotlib to show the icon predictions.
import matplotlib.pyplot as plt
# Import NumPy to build small black-and-white icon images.
import numpy as np
# Split examples into training and fair-test groups.
from sklearn.model_selection import train_test_split
# Import the nearby-neighbour voting model.
from sklearn.neighbors import KNeighborsClassifier
# Import a tool for measuring prediction accuracy.
from sklearn.metrics import accuracy_score

# Define a function that draws one small sailboat icon.
def make_boat(shift):
    # Start with an empty 16 by 16 black image.
    icon = np.zeros((16, 16))
    # Move the object a little left or right to make varied examples.
    centre = 8 + shift
    # Draw a wide boat hull near the bottom.
    icon[11:13, centre - 5:centre + 5] = 1
    # Draw a vertical mast above the hull.
    icon[5:11, centre] = 1
    # Repeat once for each row of the triangular sail.
    for row in range(6, 11):
        # Add a little more sail as the row moves downward.
        icon[row, centre + 1:centre + row - 3] = 1
    # Give the finished boat picture back to the main program.
    return icon

# Define a function that draws one small tent icon.
def make_tent(shift):
    # Start with an empty 16 by 16 black image.
    icon = np.zeros((16, 16))
    # Move the object a little left or right to make varied examples.
    centre = 8 + shift
    # Repeat once for each row of the triangular tent.
    for row in range(4, 12):
        # Work out how wide this row of the tent should be.
        half_width = (row - 3) // 2
        # Fill this row from the left edge of the tent to the right edge.
        icon[row, centre - half_width:centre + half_width + 1] = 1
    # Cut out a small doorway at the bottom of the tent.
    icon[10:12, centre - 1:centre + 2] = 0
    # Give the finished tent picture back to the main program.
    return icon

# Create a repeatable random-number helper for small icon changes.
rng = np.random.default_rng(7)
# Start an empty list for the icon pictures.
icons = []
# Start an empty list for the correct icon labels.
labels = []
# Repeat the picture-making steps for boats and tents.
for label, maker in [('boat', make_boat), ('tent', make_tent)]:
    # Make many slightly different examples of this object.
    for _ in range(80):
        # Draw one icon with a small left-or-right movement.
        icon = maker(int(rng.integers(-2, 3)))
        # Choose a few random pixels to act like image noise.
        noise = rng.random(icon.shape) < 0.02
        # Keep the icon pixels and add the tiny amount of noise.
        icons.append(np.maximum(icon, noise.astype(float)))
        # Save the correct answer for this icon.
        labels.append(label)
# Flatten each 16 by 16 picture into one row of numbers for KNN.
X = np.array(icons).reshape(len(icons), -1)
# Turn the answer list into a NumPy array.
y = np.array(labels)
# Keep one quarter of the examples secret for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=12, stratify=y)
# Let five similar training icons vote for each prediction.
model = KNeighborsClassifier(n_neighbors=5)
# Train the model using the labelled boat and tent icons.
model.fit(X_train, y_train)
# Predict the secret test icons.
predicted = model.predict(X_test)
# Calculate the percentage of correct predictions.
accuracy = accuracy_score(y_test, predicted)
# Print the score before looking at individual icons.
print('Test accuracy:', f'{accuracy:.1%}')

# Create eight small plots to make the predictions easy to inspect.
fig, axes = plt.subplots(2, 4, figsize=(7, 3.6))
# Repeat the display steps for eight secret test icons.
for ax, picture, truth, guess in zip(axes.flat, X_test[:8].reshape(-1, 16, 16), y_test[:8], predicted[:8]):
    # Draw the icon with crisp pixel edges.
    ax.imshow(picture, cmap='gray_r', interpolation='nearest')
    # Use green for a correct prediction and red for a mistake.
    colour = 'green' if truth == guess else 'crimson'
    # Label each icon with the real and predicted object.
    ax.set_title(f'True {truth}\nAI {guess}', color=colour, fontsize=9)
    # Hide axes because they do not help us read an icon.
    ax.axis('off')
# Space the icon plots so labels stay readable.
fig.tight_layout()


## Project C: Four-picture image classifier

This project joins four built-in pictures into one busy collage: coffee, a cat, an astronaut, and coins. A classifier suggests labels for the whole collage, but it cannot show where each item is. Project D introduces boxes for that job. The images are included with scikit-image; Google Colab only needs internet access the first time it downloads the pre-trained model.


In [ ]:
# Import OpenCV for image resizing and colour conversion.
import cv2
# Import NumPy for joining image tiles into a collage.
import numpy as np
# Import Matplotlib to show the collage and scores.
import matplotlib.pyplot as plt
# Import a collection of built-in example images.
from skimage import data
# Import the pre-trained classifier and its helper tools.
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions

# Load a built-in picture of a cup and spoon.
coffee_photo = data.coffee()
# Load a built-in picture of a cat.
cat_photo = data.chelsea()
# Load a built-in picture of an astronaut.
astronaut_photo = data.astronaut()
# Load a built-in picture containing many coins.
coins_photo = data.coins()

# Turn the grey coins picture into a three-colour picture like the other tiles.
coins_photo = cv2.cvtColor(coins_photo, cv2.COLOR_GRAY2RGB)
# Choose the same width and height for every small picture.
tile_size = (224, 224)
# Resize the coffee picture so it fits one collage space.
coffee_tile = cv2.resize(coffee_photo, tile_size)
# Resize the cat picture so it fits one collage space.
cat_tile = cv2.resize(cat_photo, tile_size)
# Resize the astronaut picture so it fits one collage space.
astronaut_tile = cv2.resize(astronaut_photo, tile_size)
# Resize the coins picture so it fits one collage space.
coins_tile = cv2.resize(coins_photo, tile_size)
# Join the first two pictures to make the top row.
top_row = np.hstack((coffee_tile, cat_tile))
# Join the next two pictures to make the bottom row.
bottom_row = np.hstack((astronaut_tile, coins_tile))
# Join both rows to make one busy image for the model.
image = np.vstack((top_row, bottom_row))

# Load the model weights learned from many labelled pictures.
model = MobileNetV2(weights='imagenet')
# Resize the collage to the size expected by this model.
model_input = cv2.resize(image, (224, 224)).astype('float32')
# Prepare the image numbers and add one image batch dimension.
model_input = preprocess_input(model_input[None, ...])
# Ask for the three strongest label predictions.
predictions = decode_predictions(model.predict(model_input, verbose=0), top=3)[0]
# Turn machine-style labels into labels that are easier to read.
labels = [label.replace('_', ' ') for _, label, _ in predictions]
# Keep the three confidence scores for the chart.
scores = [score for _, _, score in predictions]

# Create one space for the busy collage and one for the score chart.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Show the collage the model examined.
axes[0].imshow(image)
# Label the source image.
axes[0].set_title('Four pictures sent to one classifier')
# Hide axes because they do not help us read the image.
axes[0].axis('off')
# Draw a chart of the three most likely labels.
axes[1].barh(labels[::-1], scores[::-1], color='#3D8DFF')
# Keep confidence values on a clear 0 to 1 scale.
axes[1].set_xlim(0, 1)
# Label the chart axis.
axes[1].set_xlabel('Model confidence')
# Label the score chart.
axes[1].set_title('Top three suggestions')
# Space the picture and chart neatly.
fig.tight_layout()


## Project D: Street-scene object detector

This project downloads a street photograph and a small detector on its first run. It should find a bus, people, and traffic lights, so Google Colab needs an internet connection.


In [ ]:
# Install the object-detection library in this Colab session.
!pip -q install ultralytics
# Import Matplotlib to show the detection result.
import matplotlib.pyplot as plt
# Import the YOLO object detector.
from ultralytics import YOLO

# Store the address of a street photograph with a bus and people.
street_photo = 'https://ultralytics.com/images/bus.jpg'
# Load a small pre-trained object detector.
model = YOLO('yolov8n.pt')
# Ask the detector to find clear objects in the street photograph.
results = model(street_photo, conf=0.35, verbose=False)
# Draw the model's boxes and labels on a copy of the photograph.
annotated_bgr = results[0].plot()
# Convert OpenCV's blue-green-red order back to red-green-blue.
annotated_rgb = annotated_bgr[:, :, ::-1]

# Create a large display area for the predicted boxes.
plt.figure(figsize=(10, 6))
# Show the annotated street photograph.
plt.imshow(annotated_rgb)
# Label the detection result.
plt.title('Objects found in a street scene')
# Hide axes because they do not help us inspect the boxes.
plt.axis('off')
# Show the image with its predicted boxes.
plt.show()

# Get the readable names for each predicted class.
names = results[0].names
# Read the class number for each predicted box.
class_ids = results[0].boxes.cls.cpu().numpy().astype(int)
# Read the confidence score for each predicted box.
confidences = results[0].boxes.conf.cpu().numpy()
# Repeat once for every object the model found.
for class_id, confidence in zip(class_ids, confidences):
    # Display one object label and its confidence score.
    print(f'{names[class_id]}: {confidence:.1%}')


## Your project record

In a Markdown cell below, complete:

1. **Project title:**
2. **What I changed:**
3. **Three tests and results:**
4. **My best output:**
5. **One limitation:**

Prepare a 1–2 minute explanation. Show what your code does, not just whether it worked.


In [ ]:
# YOUR TURN
# Pick one project section above and run its code cell before changing anything.
# Hint 1: Change one setting at a time: a red HSV range, the K value, one collage tile, or the detection confidence.
# Hint 2: In Project C, try removing one tile or making one tile larger before running the model again.
# Hint 3: Use the visible mask, icon labels, confidence chart, or boxes as evidence for your explanation.
# Check: Save or screenshot your best result and write one thing that still needs improving.


## Final reflection

What would you need to improve before using your project in the real world?
